# 04 — Multi-Pair Research

**Objective:** move beyond one hand-picked pair and screen a small equity universe using only a formation period.

I compare return correlation and cointegration p-values for every pair.

In [ ]:
import itertools
import pandas as pd
import yfinance as yf

from statsmodels.tsa.stattools import coint

UNIVERSE = [
    "KO", "PEP", "JPM", "BAC", "XOM", "CVX",
    "V", "MA", "HD", "LOW", "WMT", "TGT"
]

prices = yf.download(
    UNIVERSE,
    start="2018-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)["Close"].dropna()

formation = prices.loc[:"2022-12-31"]
test = prices.loc["2023-01-01":]

print(f"Formation period: {formation.index.min().date()} to {formation.index.max().date()}")
print(f"Test period:      {test.index.min().date()} to {test.index.max().date()}")

## Screen all pairs

In [ ]:
formation_returns = formation.pct_change().dropna()
rows = []

for a, b in itertools.combinations(formation.columns, 2):
    correlation = formation_returns[a].corr(formation_returns[b])
    _, pvalue, _ = coint(formation[a], formation[b])

    rows.append({
        "asset_1": a,
        "asset_2": b,
        "return_corr": correlation,
        "coint_pvalue": pvalue
    })

pair_results = (
    pd.DataFrame(rows)
    .sort_values(["coint_pvalue", "return_corr"], ascending=[True, False])
)

pair_results.head(15)

## Candidate pairs

I use a simple formation-period filter of correlation above `0.5` and cointegration p-value below `0.05`.

The filter is only used to identify candidates; significance alone does not imply a profitable strategy.

In [ ]:
candidate_pairs = pair_results[
    (pair_results["return_corr"] > 0.5) &
    (pair_results["coint_pvalue"] < 0.05)
].copy()

candidate_pairs